In [68]:
import csv
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup

In [69]:
base_url = 'https://www.braceletbook.com/'
patterns = 'patterns/'
data_directory = Path.expanduser(Path('~/data/braceletbook/'))

In [70]:
class NotFoundException(Exception):
  pass


def get_page(url, cache_path):
  cache_path = Path(cache_path)
  if cache_path.exists():
    return cache_path.read_text()
  r = requests.get(
    url,
    headers={
      "User-Agent": 'braceletbook-scraper (personal use: extended search, contact: dev@nategrigg.com)'
    }
  )

  if r.status_code == 404:
    # what to throw? custom exception?
    raise NotFoundException(f'Page not found: {url}')
  cache_path.write_text(r.text)
  time.sleep(1)
  return r.text

In [71]:
def page_pull_loop():
  page_start = 6499
  page_end = 7000

  detailed_file = data_directory.joinpath(f'details-{page_start}-{page_end - 1}.csv')
  details = {}

  with open(detailed_file, 'w') as f:
    csv_writer = csv.writer(f)
    csv_writer.writerow(['pattern_id', 'string_count', 'color_count'])

    for page_no in range(page_start, page_end):
      page_url = base_url + patterns + f'page-{page_no}'
      try:
        text = get_page(
          page_url,
          cache_path=data_directory.joinpath(Path(f'page-{page_no:04}.html'))
        )
      except NotFoundException:
        break

      soup = BeautifulSoup(text, 'html.parser')

      for pattern_id in soup.find_all('a', class_='id'):
        details[pattern_id] = {}
        string_icon = soup.find('div', class_='strings_icon')
        if not string_icon:
          continue
        string_count_span = string_icon.find_next_sibling('span', class_='caption')
        string_count = int(string_count_span.text)
        details[pattern_id]['string_count'] = string_count

        color_icon = soup.find('div', class_='colors_icon')
        if not color_icon:
          continue
        color_count_span = color_icon.find_next_sibling('span', class_='caption')
        color_count = int(color_count_span.text)
        details[pattern_id]['color_count'] = color_count

        pattern_details = [
          pattern_id.text.strip('#'),
          string_count,
          color_count
        ]
        csv_writer.writerow(pattern_details)


In [72]:
def filter_pass(input_file, output_file):
  if not input_file:
    return

  if not input_file.exists():
    return

  page_soup = BeautifulSoup(input_file.read_text(), 'html.parser')
  pattern_divs = page_soup.find_all('div', class_='patterns_item')
  for div in pattern_divs:
    pattern_link = div.find('a', class_='id')
    if pattern_link and pattern_link.has_attr('href'):
      if 'patterns/normal' in pattern_link.attrs['href']:
        output_file.write(str(div))
        output_file.flush()


def write_head(output_file):
  output_file.write('<html><head>')
  output_file.write('<title>Non Alpha Cache</title>')
  output_file.write(
    """  <link rel="stylesheet"
                      href="https://static.braceletbookcdn.com/cache/css/output.72ab83ee1321.css"
                      type="text/css">"""
  )
  output_file.write('</head><body>')
  output_file.flush()


def write_foot(output_file):
  output_file.write('</body></html>')


def find_normal_patterns(output_file_name='non_alpha_cache.html', page_range=(0,)):
  with open(data_directory.joinpath(output_file_name), 'w') as output_file:
    write_head(output_file)

    for page_number in page_range:
      input_file = data_directory.joinpath(f'page-{page_number:04}.html')
      filter_pass(input_file, output_file)

    write_foot(output_file)

this_page_range = range(1,10)
find_normal_patterns('non_alpha_cache.html', this_page_range)
